# Получение и сохранение показаний ESP32

**DHT22 → ESP32 → USB → Python → SQLite.** MQTT пока не используется.

Закрой монитор порта Arduino IDE. Выбери ядро **microclimate** и запускай ячейки сверху вниз. База уже включена в проект. При создании новой базы сначала выполни `01_database.ipynb`. Учебные пробы и искусственные измерения в этот блокнот не входят.

## 1. Импорты и настройки
Укажи порт своей ESP32. Скорость совпадает с `Serial.begin(115200)`. Сбор ограничен пятью парами; ожидание корректной строки — 15 секунд.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "db.py").is_file():
    project_root = project_root.parent
if not (project_root / "db.py").is_file():
    raise FileNotFoundError("Открой блокнот из папки microclimate или microclimate/notebooks.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import sqlite3
import serial
from db import DB_PATH, get_conn
from receiver import collect_pairs
from repo import recent_measurements, recent_alerts

port = "/dev/cu.usbserial-10"
baudrate = 115200
pair_count = 5
idle_timeout = 15

## 2. Подключение к существующей базе
При неверном пути пустая база автоматически не создаётся. Проверка внешних ключей включается при подключении.

In [ ]:
if "conn" in globals():
    conn.close()
conn = get_conn()
print("База:", DB_PATH)

## 3. Сбор с проверкой порогов
Каждый запуск получает новые показания. Два измерения и предупреждения сохраняются одной транзакцией на пару. Служебные и повреждённые строки пропускаются. При ошибке уже сохранённые пары остаются в базе. Время — момент получения на компьютере в UTC.

In [ ]:
try:
    saved = collect_pairs(conn, port, baudrate, pair_count, idle_timeout)
    print("Сбор завершён. Сохранено пар:", saved)
except (serial.SerialException, OSError, TimeoutError, ValueError,
        sqlite3.IntegrityError, sqlite3.OperationalError) as error:
    print("Сбор остановлен:", error)

## 4. Последние измерения и предупреждения
Просмотр доступен и без подключённой платы. Равенство порогу допустимо; `below` означает ниже границы, `above` — выше. Отсутствие действующего порога выводится при сборе.

In [ ]:
print("Последние измерения (UTC):")
for row in recent_measurements(conn):
    print(dict(row))

print("\nПоследние предупреждения (UTC):")
alerts = recent_alerts(conn)
if not alerts:
    print("Предупреждений нет.")
for row in alerts:
    print(dict(row))

## 5. Завершение
Порт закрывается автоматически после сбора, соединение с базой закрываем явно. Для нового сеанса выполни шаг 2, затем шаги 3–5.

In [ ]:
conn.close()
print("Соединение закрыто.")